# Distributed Inference at Scale

এখানে কোনো multi-GPU hardware প্রয়োজন হয় না (বা ব্যবহারও হয় না) — দুটি demo-ই বাস্তব distributed-inference প্রভাবের সৎ, formula/Monte-Carlo-ভিত্তিক simulation, স্পষ্টভাবে সেভাবেই label করা।

দুটি demo:

1. **Pipeline-parallelism bubble sweep:** প্রকৃত `bubble_fraction = (num_stages - 1) / num_micro_batches` formula-টি `num_micro_batches`-এর বিভিন্ন মান এবং `num_stages`-এর কয়েকটি মান জুড়ে গণনা করে, মূর্তভাবে দেখায় যে `num_micro_batches=1` (একটি একক request-এর decode step — একটি token, pipeline করার কিছু নেই) একটি বিশাল bubble রেখে যায়, অন্যদিকে অনেকগুলো একযোগে চলমান micro-batch (অনেকগুলো একযোগে চলা decode request একসাথে batch করা, অর্থাৎ continuous batching) এটিকে শূন্যের দিকে নামিয়ে আনে।
2. **Expert-parallelism idle-GPU Monte Carlo simulation:** decode-কালের বিভিন্ন batch size-এর জন্য, batch-এর প্রতিটি token-এর জন্য সত্যিই random top-k expert routing সিদ্ধান্ত sample করে (প্রকৃত randomness, কোনো hardcoded সংখ্যা নয়) এবং সেই ধাপে কমপক্ষে একটি token পাওয়া expert-এর (এবং তাই GPU-র, প্রতি GPU-তে একটি expert ধরে নিয়ে) প্রকৃত ভগ্নাংশ মাপে — দেখায় যে ছোট decode batch অনেক expert/GPU-কে অলস রাখে, অন্যদিকে বড় batch প্রায় সবগুলোকে saturate করে।

Runtime: CPU-তে এক মিনিটেরও অনেক কম (বিশুদ্ধ arithmetic, সাথে কয়েক হাজার সস্তা random-sampling trial)।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান।

In [ ]:
import random

random.seed(0)

## 1. Pipeline parallelism: bubble formula, সত্যিই গণনা করে sweep করা

এখানে Lesson README section 3-এর হুবহু formula — `bubble_fraction = (num_stages - 1) / num_micro_batches` — `num_micro_batches`-এর একটি sweep এবং দুটি `num_stages` মান (4 ও 8) জুড়ে গণনা করা হয়। Demo-টি একটি table print করে, তারপর `num_stages=8`-এ একটি একক request-এর decode step (`num_micro_batches=1`) বনাম 64 টি একযোগে চলমান decode step-এর তুলনা করে দেখায় কেন pipeline-parallel inference throughput continuous batching-এর উপর নির্ভর করে।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Pipeline parallelism: bubble formula, সত্যিই গণনা করা, num_micro_batches
#    জুড়ে sweep করা -- হুবহু Lesson README section 3-এর formula।
# ---------------------------------------------------------------------------

def bubble_fraction(num_stages, num_micro_batches):
    """প্রকৃত formula: bubble_fraction = (num_stages - 1) / num_micro_batches।
    num_micro_batches=1 মানে pipeline-এর মধ্য দিয়ে ঠিক একটি কাজের ইউনিট
    চলছে (একটি একক request-এর একক decode-step token) -- সবচেয়ে খারাপ
    ক্ষেত্র। বড় num_micro_batches মানে আরও স্বাধীন কাজের ইউনিট (যেমন আরও
    একযোগে চলমান request-এর decode step) যা প্রতিটি stage-কে একসাথে ব্যস্ত
    রাখে।"""
    return (num_stages - 1) / num_micro_batches


def pipeline_bubble_demo():
    print("=" * 70)
    print("1. PIPELINE PARALLELISM: THE BUBBLE FRACTION, SWEPT FOR REAL")
    print("=" * 70)

    stage_counts = [4, 8]
    micro_batch_counts = [1, 2, 4, 8, 16, 64]

    print("\nbubble_fraction = (num_stages - 1) / num_micro_batches")
    print("(fraction of every stage's time spent idle, waiting on the pipeline)\n")

    header = f"{'num_micro_batches':>18}" + "".join(f"{'stages=' + str(s):>16}" for s in stage_counts)
    print(header)
    for m in micro_batch_counts:
        row = f"{m:>18}"
        for s in stage_counts:
            frac = bubble_fraction(s, m)
            row += f"{frac:>15.3f} "
        print(row)

    single_token_bubble = bubble_fraction(8, 1)
    many_concurrent_bubble = bubble_fraction(8, 64)
    print(f"\nAt num_stages=8:")
    print(f"  num_micro_batches=1  (one request, one decode-step token): "
          f"bubble_fraction = {single_token_bubble:.3f}  -> {single_token_bubble*100:.1f}% of every stage's time IDLE")
    print(f"  num_micro_batches=64 (64 concurrent in-flight decode steps): "
          f"bubble_fraction = {many_concurrent_bubble:.3f}  -> {many_concurrent_bubble*100:.1f}% idle")
    print("\n-> A single request's decode step has exactly ONE token to pipeline -- there is no")
    print("   'more micro-batches' available within that one request to fill the bubble with.")
    print("   The bubble only shrinks toward zero when the SERVER supplies many concurrent")
    print("   requests' decode steps as additional micro-batches flowing through the same")
    print("   pipeline stages simultaneously -- exactly why pipeline-parallel inference")
    print("   throughput depends on continuous batching keeping enough requests in flight.")


pipeline_bubble_demo()

## 2. Expert parallelism: decode-কালের idle-GPU ঝুঁকির Monte Carlo simulation

একটি batch-এর প্রতিটি token স্বাধীনভাবে expert-এর একটি random top-k subset sample করে (uniform random routing — বহু draw-এর গড়ে একটি load-balanced router-এর যুক্তিসঙ্গত stand-in)। `simulate_expert_utilization` মাপে সেই ধাপে কমপক্ষে একটি token পাওয়া expert-এর প্রকৃত ভগ্নাংশ কত। Demo-টি 64 টি expert (প্রতি GPU-তে একটি) ও top-2 routing দিয়ে বিভিন্ন batch size-এর জন্য গড় স্পর্শিত ভগ্নাংশ ও অলস expert-সংখ্যা print করে, এবং `batch_size=1` বনাম `batch_size=256` তুলনা করে।

In [ ]:
# ---------------------------------------------------------------------------
# 2. Expert parallelism: decode-কালের idle-GPU ঝুঁকির Monte Carlo simulation।
#    একটি batch-এর প্রতিটি token স্বাধীনভাবে expert-এর একটি random top-k
#    subset sample করে (uniform random routing -- বহু draw-এর গড়ে একটি
#    load-balanced router-এর যুক্তিসঙ্গত stand-in); আমরা সেই ধাপে কমপক্ষে একটি
#    token দ্বারা স্পর্শিত expert-এর প্রকৃত ভগ্নাংশ মাপি।
# ---------------------------------------------------------------------------

def simulate_expert_utilization(num_experts, experts_per_token, batch_size, num_trials=2000):
    """batch_size আকারের num_trials টি স্বাধীন random batch জুড়ে, কমপক্ষে
    একটি token পাওয়া expert-এর গড় ভগ্নাংশ ফেরত দেয়। প্রতিটি token
    num_experts-এর মধ্য থেকে uniformly random-ভাবে experts_per_token টি
    আলাদা expert sample করে (প্রতিটি trial-এ, প্রতিটি token-এ একটি প্রকৃত
    random draw -- কোনো hardcoded/দাবি করা সংখ্যা নয়)।"""
    total_fraction = 0.0
    for _ in range(num_trials):
        touched = set()
        for _ in range(batch_size):
            chosen = random.sample(range(num_experts), experts_per_token)
            touched.update(chosen)
        total_fraction += len(touched) / num_experts
    return total_fraction / num_trials


def expert_parallelism_demo():
    print("\n" + "=" * 70)
    print("2. EXPERT PARALLELISM: MONTE CARLO EXPERT/GPU UTILIZATION PER STEP")
    print("=" * 70)

    num_experts = 64
    experts_per_token = 2   # top-k routing, k=2 (Mixtral-style)
    batch_sizes = [1, 4, 16, 64, 256]

    print(f"\n{num_experts} experts total (one expert per GPU), top-{experts_per_token} routing per token.")
    print("Each trial: sample REAL random top-k expert choices for every token in the batch,")
    print("then measure what fraction of the 64 experts (GPUs) got at least one token.\n")

    print(f"{'batch_size':>12}{'avg fraction of experts touched':>34}{'avg experts IDLE that step':>30}")
    for bs in batch_sizes:
        frac = simulate_expert_utilization(num_experts, experts_per_token, bs)
        idle_experts = num_experts * (1 - frac)
        print(f"{bs:>12}{frac:>34.3f}{idle_experts:>30.1f}")

    frac_1 = simulate_expert_utilization(num_experts, experts_per_token, 1)
    frac_256 = simulate_expert_utilization(num_experts, experts_per_token, 256)
    print(f"\nAt batch_size=1   (a single decode-step token): only {frac_1*100:.1f}% of experts touched "
          f"-> {num_experts*(1-frac_1):.1f} of {num_experts} GPUs sit idle that step.")
    print(f"At batch_size=256 (many concurrent decode requests batched together): "
          f"{frac_256*100:.1f}% of experts touched -> only {num_experts*(1-frac_256):.1f} GPUs idle.")
    print("\n-> With a small decode batch, the handful of tokens in flight route to only a few")
    print("   of the 64 experts -- every GPU holding an unrouted-to expert does no useful work")
    print("   that step, a real distinct failure mode from the pipeline bubble above (whole")
    print("   devices going unused for a step, not stages waiting on data). Large batches")
    print("   route enough independent tokens that, statistically, nearly every expert gets")
    print("   touched -- which is exactly why MoE inference serving needs large enough batches")
    print("   (or clever expert placement/replication) to keep its expert-holding GPUs busy.")


expert_parallelism_demo()

## সবগুলো demo একসাথে

`main()` উপরের দুটি demo — pipeline-bubble sweep এবং expert-parallelism Monte Carlo simulation — পরপর এক সাথে চালায়। উপরের cell-গুলোতে প্রতিটি demo ইতিমধ্যে চলেছে, তাই demo দুবার না চালাতে call-টি comment করে রাখা হয়েছে।

In [ ]:
def main():
    pipeline_bubble_demo()
    expert_parallelism_demo()


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন